In [1]:
import pandas as pd

# Day 4 — Step 1.1
# Synthetic Security Master (not real market data)

security_master = pd.DataFrame([
    {
        "security_id": 1001,
        "ticker": "ALPHA",
        "listing_date": "2010-01-04",
        "delisting_date": None,
        "exchange": "NYSE",
        "share_type": "COMMON",
    },
    {
        "security_id": 1002,
        "ticker": "BETA",
        "listing_date": "2011-06-01",
        "delisting_date": "2018-09-28",
        "exchange": "NASDAQ",
        "share_type": "COMMON",
    },
    {
        "security_id": 1003,
        "ticker": "GAMMA",
        "listing_date": "2020-04-15",
        "delisting_date": None,
        "exchange": "NASDAQ",
        "share_type": "COMMON",
    },
    {
        "security_id": 1004,
        "ticker": "DELTA",
        "listing_date": "2012-03-01",
        "delisting_date": None,
        "exchange": "NYSE",
        "share_type": "ETF",
    },
    {
        "security_id": 1005,
        "ticker": "EPSILON",
        "listing_date": "2013-01-02",
        "delisting_date": None,
        "exchange": "OTC",
        "share_type": "COMMON",
    },
])

for column in ["listing_date", "delisting_date"]:
    security_master[column] = pd.to_datetime(
        security_master[column]
    )

display(security_master)

print("Shape:", security_master.shape)
print(
    "Duplicate security IDs:",
    security_master["security_id"].duplicated().sum()
)

,security_id,ticker,listing_date,delisting_date,exchange,share_type
0,1001,ALPHA,2010-01-04,NaT,NYSE,COMMON
1,1002,BETA,2011-06-01,2018-09-28,NASDAQ,COMMON
2,1003,GAMMA,2020-04-15,NaT,NASDAQ,COMMON
3,1004,DELTA,2012-03-01,NaT,NYSE,ETF
4,1005,EPSILON,2013-01-02,NaT,OTC,COMMON


Shape: (5, 6)
Duplicate security IDs: 0


In [2]:
# Day 4 — Step 2.1
# Point-in-Time Universe Eligibility Prototype

def get_pit_universe(security_master, as_of_date):
    as_of_date = pd.Timestamp(as_of_date)

    # 1. Already listed by the selection date
    listed = (
        security_master["listing_date"] <= as_of_date
    )

    # 2. Not delisted before the selection date
    active = (
        security_master["delisting_date"].isna()
        | (security_master["delisting_date"] >= as_of_date)
    )

    # 3. Eligible US exchanges
    valid_exchange = security_master["exchange"].isin(
        ["NYSE", "NASDAQ", "NYSE AMERICAN"]
    )

    # 4. Common shares only
    valid_share_type = (
        security_master["share_type"] == "COMMON"
    )

    eligible = (
        listed
        & active
        & valid_exchange
        & valid_share_type
    )

    return (
        security_master.loc[
            eligible,
            ["security_id", "ticker",
             "listing_date", "delisting_date"]
        ]
        .sort_values("security_id")
        .reset_index(drop=True)
    )


# Test historical universe construction
for date in ["2015-06-30", "2019-06-30", "2021-06-30"]:
    universe = get_pit_universe(
        security_master, date
    )

    print(f"\nUniverse as of {date}:")
    print(universe["ticker"].tolist())


Universe as of 2015-06-30:
['ALPHA', 'BETA']

Universe as of 2019-06-30:
['ALPHA']

Universe as of 2021-06-30:
['ALPHA', 'GAMMA']


In [3]:
# Day 4 — Step 2.2
# Boundary-date checks

test_dates = [
    "2011-05-31",  # Before BETA's listing
    "2011-06-01",  # BETA listing date
    "2018-09-28",  # BETA delisting date
    "2018-10-01",  # After BETA's delisting
    "2020-04-15",  # GAMMA listing date
]

for date in test_dates:
    universe = get_pit_universe(
        security_master, date
    )
    print(
        date,
        "->",
        universe["ticker"].tolist()
    )


2011-05-31 -> ['ALPHA']
2011-06-01 -> ['ALPHA', 'BETA']
2018-09-28 -> ['ALPHA', 'BETA']
2018-10-01 -> ['ALPHA']
2020-04-15 -> ['ALPHA', 'GAMMA']


In [4]:
# Day 4 — Step 3.1
# Automated PIT Universe Tests

def get_tickers(as_of_date):
    universe = get_pit_universe(
        security_master,
        as_of_date
    )
    return set(universe["ticker"])


# Test 1: Historical eligibility
assert get_tickers("2015-06-30") == {
    "ALPHA", "BETA"
}

# Test 2: Delisted security remains
# in the historical universe
assert "BETA" in get_tickers("2015-06-30")

# Test 3: Future IPO must not appear
assert "GAMMA" not in get_tickers("2015-06-30")

# Test 4: ETF must be excluded
assert "DELTA" not in get_tickers("2021-06-30")

# Test 5: OTC security must be excluded
assert "EPSILON" not in get_tickers("2021-06-30")

# Test 6: Listing-date boundary
assert "BETA" not in get_tickers("2011-05-31")
assert "BETA" in get_tickers("2011-06-01")

# Test 7: Delisting-date boundary
assert "BETA" in get_tickers("2018-09-28")
assert "BETA" not in get_tickers("2018-10-01")

print("All 7 PIT universe tests passed.")

All 7 PIT universe tests passed.


In [5]:
# Day 4 — Step 3.2
# Deliberately introduce survivorship bias

# Incorrect approach:
# Remove every security with a delisting date
survivors_only = security_master.loc[
    security_master["delisting_date"].isna()
].copy()

correct_universe = get_pit_universe(
    security_master, "2015-06-30"
)

biased_universe = get_pit_universe(
    survivors_only, "2015-06-30"
)

print(
    "Correct 2015 universe:",
    correct_universe["ticker"].tolist()
)

print(
    "Biased 2015 universe:",
    biased_universe["ticker"].tolist()
)

missing = (
    set(correct_universe["ticker"])
    - set(biased_universe["ticker"])
)

print("Historical securities lost:", missing)

assert "BETA" in missing

print("Survivorship-bias test demonstrated.")

Correct 2015 universe: ['ALPHA', 'BETA']
Biased 2015 universe: ['ALPHA']
Historical securities lost: {'BETA'}
Survivorship-bias test demonstrated.


In [6]:
# Day 5 — Step 1.1
# Synthetic Daily Price and Volume Data

import pandas as pd

# Use business days as a simplified synthetic calendar.
# This is NOT a complete NYSE trading calendar.
dates = pd.bdate_range(
    start="2021-01-04",
    periods=80
)

# Synthetic security IDs from Day 4:
# 1001 = ALPHA
# 1002 = BETA (delisted in 2018)
# 1003 = GAMMA
# 1004 = DELTA (ETF)
# 1005 = EPSILON (OTC)

synthetic_specs = {
    1001: {"close": 100.0, "volume": 200_000},
    1002: {"close": 50.0,  "volume": 500_000},
    1003: {"close": 4.0,   "volume": 5_000_000},
    1004: {"close": 80.0,  "volume": 300_000},
    1005: {"close": 20.0,  "volume": 800_000},
}

records = []

for date in dates:
    for security_id, spec in synthetic_specs.items():
        records.append({
            "date": date,
            "security_id": security_id,
            "close": spec["close"],
            "volume": spec["volume"],
        })

daily_prices = pd.DataFrame(records)

daily_prices["dollar_volume"] = (
    daily_prices["close"] * daily_prices["volume"]
)

daily_prices = daily_prices.sort_values(
    ["security_id", "date"]
).reset_index(drop=True)

display(daily_prices.head(10))

print("Shape:", daily_prices.shape)
print(
    "Date range:",
    daily_prices["date"].min().date(),
    "to",
    daily_prices["date"].max().date()
)

print(
    "Duplicate security-date pairs:",
    daily_prices.duplicated(
        ["security_id", "date"]
    ).sum()
)

,date,security_id,close,volume,dollar_volume
0,2021-01-04,1001,100.0,200000,20000000.0
1,2021-01-05,1001,100.0,200000,20000000.0
2,2021-01-06,1001,100.0,200000,20000000.0
3,2021-01-07,1001,100.0,200000,20000000.0
4,2021-01-08,1001,100.0,200000,20000000.0
5,2021-01-11,1001,100.0,200000,20000000.0
6,2021-01-12,1001,100.0,200000,20000000.0
7,2021-01-13,1001,100.0,200000,20000000.0
8,2021-01-14,1001,100.0,200000,20000000.0
9,2021-01-15,1001,100.0,200000,20000000.0


Shape: (400, 5)
Date range: 2021-01-04 to 2021-04-23
Duplicate security-date pairs: 0


In [7]:
# Day 5 — Step 2.1
# Point-in-Time Price Eligibility

def get_pit_price_universe(
    security_master,
    daily_prices,
    as_of_date,
    min_price=5.0
):
    as_of_date = pd.Timestamp(as_of_date)

    # Step 1: Historical security eligibility
    universe = get_pit_universe(
        security_master,
        as_of_date
    )

    # Step 2: Only use price records at or before cutoff
    historical_prices = daily_prices.loc[
        daily_prices["date"] <= as_of_date
    ].copy()

    # Step 3: Find the latest available price
    # for each security by the cutoff
    latest_prices = (
        historical_prices
        .sort_values(["security_id", "date"])
        .drop_duplicates(
            subset="security_id",
            keep="last"
        )
    )

    # Step 4: Attach price information to the PIT universe
    universe = universe.merge(
        latest_prices[
            ["security_id", "date", "close"]
        ],
        on="security_id",
        how="left",
        validate="one_to_one"
    )

    # Step 5: Apply minimum price threshold
    universe["price_eligible"] = (
        universe["close"].notna()
        & (universe["close"] >= min_price)
    )

    return universe


# Test on a date covered by our synthetic data
result = get_pit_price_universe(
    security_master,
    daily_prices,
    as_of_date="2021-04-15",
    min_price=5.0
)

display(result)


,security_id,ticker,listing_date,delisting_date,date,close,price_eligible
0,1001,ALPHA,2010-01-04,NaT,2021-04-15,100.0,True
1,1003,GAMMA,2020-04-15,NaT,2021-04-15,4.0,False


In [8]:
# Day 5 — Step 2.2
# Automated Price Eligibility Tests

result = get_pit_price_universe(
    security_master,
    daily_prices,
    "2021-04-15"
)

# Test 1: Correct securities after PIT screening
assert set(result["ticker"]) == {
    "ALPHA", "GAMMA"
}

# Test 2: ALPHA passes minimum price
alpha = result.loc[
    result["ticker"] == "ALPHA"
].iloc[0]

assert alpha["close"] == 100.0
assert bool(alpha["price_eligible"])

# Test 3: GAMMA fails minimum price
gamma = result.loc[
    result["ticker"] == "GAMMA"
].iloc[0]

assert gamma["close"] == 4.0
assert not bool(gamma["price_eligible"])

# Test 4: Delisted BETA cannot re-enter
assert "BETA" not in result["ticker"].values

# Test 5: No future price dates were selected
assert (
    result["date"].dropna()
    <= pd.Timestamp("2021-04-15")
).all()

print("All 5 PIT price eligibility tests passed.")

All 5 PIT price eligibility tests passed.


In [9]:
# Day 5 — Step 3.1
# Point-in-Time Median Dollar Volume

def get_pit_liquidity(
    daily_prices,
    as_of_date,
    window=60
):
    as_of_date = pd.Timestamp(as_of_date)

    # Only historical observations
    history = daily_prices.loc[
        daily_prices["date"] <= as_of_date
    ].copy()

    # One row per security per date is required
    if history.duplicated(["security_id", "date"]).any():
        raise ValueError("Duplicate security-date records")

    # Compute dollar volume using compatible units
    history["dollar_volume"] = (
        history["close"] * history["volume"]
    )

    # Sort before taking the trailing window
    history = history.sort_values(
        ["security_id", "date"]
    )

    # Keep the most recent N observations per security
    trailing = (
        history.groupby("security_id")
        .tail(window)
    )

    summary = (
        trailing.groupby("security_id")
        .agg(
            median_dollar_volume=(
                "dollar_volume", "median"
            ),
            observation_count=(
                "dollar_volume", "count"
            )
        )
        .reset_index()
    )

    # Require a complete history window
    summary.loc[
        summary["observation_count"] < window,
        "median_dollar_volume"
    ] = float("nan")

    return summary


liquidity = get_pit_liquidity(
    daily_prices,
    as_of_date="2021-04-15",
    window=60
)

display(liquidity)


,security_id,median_dollar_volume,observation_count
0,1001,20000000.0,60
1,1002,25000000.0,60
2,1003,20000000.0,60
3,1004,24000000.0,60
4,1005,16000000.0,60


In [10]:
# Day 5 — Step 3.2
# Combined Price and Liquidity Eligibility

def get_pit_tradable_universe(
    security_master,
    daily_prices,
    as_of_date,
    min_price=5.0,
    min_median_dollar_volume=10_000_000,
    window=60
):
    # Historical security and price eligibility
    universe = get_pit_price_universe(
        security_master,
        daily_prices,
        as_of_date,
        min_price=min_price
    )

    # Historical liquidity summary
    liquidity = get_pit_liquidity(
        daily_prices,
        as_of_date,
        window=window
    )

    # Merge by stable security ID
    universe = universe.merge(
        liquidity,
        on="security_id",
        how="left",
        validate="one_to_one"
    )

    universe["liquidity_eligible"] = (
        universe["median_dollar_volume"].notna()
        & (
            universe["median_dollar_volume"]
            >= min_median_dollar_volume
        )
    )

    universe["eligible"] = (
        universe["price_eligible"]
        & universe["liquidity_eligible"]
    )

    return universe


result = get_pit_tradable_universe(
    security_master,
    daily_prices,
    as_of_date="2021-04-15"
)

display(
    result[
        [
            "ticker",
            "close",
            "median_dollar_volume",
            "observation_count",
            "price_eligible",
            "liquidity_eligible",
            "eligible"
        ]
    ]
)


,ticker,close,median_dollar_volume,observation_count,price_eligible,liquidity_eligible,eligible
0,ALPHA,100.0,20000000.0,60,True,True,True
1,GAMMA,4.0,20000000.0,60,False,True,False


In [11]:
# Day 5 — Step 3.3
# Incomplete liquidity window test

early_liquidity = get_pit_liquidity(
    daily_prices,
    as_of_date="2021-01-15",
    window=60
)

display(early_liquidity)

# None of the securities has 60 observations yet
assert early_liquidity[
    "median_dollar_volume"
].isna().all()

print("Incomplete-window test passed.")

,security_id,median_dollar_volume,observation_count
0,1001,NaN,10
1,1002,NaN,10
2,1003,NaN,10
3,1004,NaN,10
4,1005,NaN,10


Incomplete-window test passed.


In [12]:
# Day 5 — Step 4.1
# Automated PIT Price & Liquidity Tests

as_of = "2021-04-15"

result = get_pit_tradable_universe(
    security_master,
    daily_prices,
    as_of
)

# Test 1: Only historically valid securities
assert set(result["ticker"]) == {"ALPHA", "GAMMA"}

# Test 2: Only ALPHA passes both filters
eligible = set(result.loc[
    result["eligible"], "ticker"
])
assert eligible == {"ALPHA"}

# Test 3: Dollar-volume calculations
mdv = result.set_index("ticker")["median_dollar_volume"]
assert mdv["ALPHA"] == 20_000_000
assert mdv["GAMMA"] == 20_000_000

# Test 4: Complete 60-observation windows
assert (result["observation_count"] == 60).all()

# Test 5: Insufficient price history
early = get_pit_tradable_universe(
    security_master,
    daily_prices,
    "2021-01-15"
)
assert not early["eligible"].any()

# Test 6: Future price changes must not affect
# the historical selection result
modified_prices = daily_prices.copy()

future_mask = (
    (modified_prices["security_id"] == 1001)
    & (modified_prices["date"] > pd.Timestamp(as_of))
)

modified_prices.loc[future_mask, "close"] = 1.0

modified_result = get_pit_tradable_universe(
    security_master,
    modified_prices,
    as_of
)

cols = [
    "security_id", "close",
    "median_dollar_volume", "eligible"
]

pd.testing.assert_frame_equal(
    result[cols].reset_index(drop=True),
    modified_result[cols].reset_index(drop=True)
)

# Test 7: Delisted BETA must stay excluded
assert "BETA" not in result["ticker"].values

print("All 7 Day 5 PIT eligibility tests passed.")


All 7 Day 5 PIT eligibility tests passed.


In [13]:
# Week 1 Day 6 — Step 1.2
# Integration Summary

print("=" * 52)
print("EQUITY ALPHA RESEARCH — WEEK 1 INTEGRATION CHECK")
print("=" * 52)

# Security master integrity
assert security_master["security_id"].is_unique
print("[PASS] Unique security identifiers")

# Synthetic daily market data integrity
assert not daily_prices.duplicated(
    ["security_id", "date"]
).any()
print("[PASS] Unique security-date observations")

# Historical universe reconstruction
universe_2015 = get_pit_universe(
    security_master,
    "2015-06-30"
)

assert set(universe_2015["ticker"]) == {
    "ALPHA", "BETA"
}
print("[PASS] Historical universe includes later-delisted BETA")

# Future IPO exclusion
assert "GAMMA" not in universe_2015["ticker"].values
print("[PASS] Future IPO excluded")

# Combined price and liquidity eligibility
tradable = get_pit_tradable_universe(
    security_master,
    daily_prices,
    "2021-04-15"
)

eligible_tickers = set(
    tradable.loc[tradable["eligible"], "ticker"]
)

assert eligible_tickers == {"ALPHA"}
print("[PASS] Combined PIT price and liquidity filters")

# Insufficient liquidity history
early = get_pit_tradable_universe(
    security_master,
    daily_prices,
    "2021-01-15"
)

assert not early["eligible"].any()
print("[PASS] Incomplete liquidity windows excluded")

print("=" * 52)
print("WEEK 1 INTEGRATION CHECKS PASSED")
print("=" * 52)


EQUITY ALPHA RESEARCH — WEEK 1 INTEGRATION CHECK
[PASS] Unique security identifiers
[PASS] Unique security-date observations
[PASS] Historical universe includes later-delisted BETA
[PASS] Future IPO excluded
[PASS] Combined PIT price and liquidity filters
[PASS] Incomplete liquidity windows excluded
WEEK 1 INTEGRATION CHECKS PASSED


In [14]:
# Day 6 — Step 2.1
# Duplicate Security-Date Data Quality Test

# Create a copy to avoid modifying the clean dataset
prices_with_duplicates = daily_prices.copy()

# Deliberately duplicate one existing row
duplicate_row = daily_prices.iloc[[0]].copy()

prices_with_duplicates = pd.concat(
    [prices_with_duplicates, duplicate_row],
    ignore_index=True
)

# Verify that the synthetic anomaly exists
duplicate_count = prices_with_duplicates.duplicated(
    ["security_id", "date"]
).sum()

print("Duplicate security-date pairs:", duplicate_count)

assert duplicate_count == 1

# The existing liquidity function should reject this input
try:
    get_pit_liquidity(
        prices_with_duplicates,
        as_of_date="2021-04-15",
        window=60
    )
except ValueError as e:
    assert "Duplicate security-date records" in str(e)
    print("[PASS] Duplicate records correctly rejected")
else:
    raise AssertionError(
        "FAIL: Duplicate records were not detected"
    )


Duplicate security-date pairs: 1
[PASS] Duplicate records correctly rejected


In [15]:
# Day 6 — Step 2.2
# Missing and Invalid Price Tests

import numpy as np

as_of = pd.Timestamp("2021-04-15")

invalid_cases = {
    "Missing price": np.nan,
    "Zero price": 0.0,
    "Negative price": -100.0,
}

for case_name, bad_price in invalid_cases.items():

    # Always start from the clean dataset
    bad_data = daily_prices.copy()

    # Modify ALPHA's price on the selection date
    mask = (
        (bad_data["security_id"] == 1001)
        & (bad_data["date"] == as_of)
    )

    assert mask.sum() == 1

    bad_data.loc[mask, "close"] = bad_price

    result = get_pit_price_universe(
        security_master,
        bad_data,
        as_of_date=as_of,
        min_price=5.0
    )

    alpha = result.loc[
        result["ticker"] == "ALPHA"
    ].iloc[0]

    print(
        f"{case_name}: "
        f"close={alpha['close']}, "
        f"eligible={alpha['price_eligible']}"
    )

    assert not bool(alpha["price_eligible"])

print("[PASS] All three invalid-price cases excluded")


Missing price: close=nan, eligible=False
Zero price: close=0.0, eligible=False
Negative price: close=-100.0, eligible=False
[PASS] All three invalid-price cases excluded


In [16]:
# Day 6 — Step 2.2.2
# Basic Market Data Quality Validator

def validate_daily_prices(data):
    issues = []

    if data.duplicated(["security_id", "date"]).any():
        issues.append("Duplicate security-date records")

    if data["close"].isna().any():
        issues.append("Missing closing prices")

    if (data["close"].dropna() <= 0).any():
        issues.append("Non-positive closing prices")

    if data["volume"].isna().any():
        issues.append("Missing trading volumes")

    if (data["volume"].dropna() < 0).any():
        issues.append("Negative trading volumes")

    return issues


# Clean data should pass
clean_issues = validate_daily_prices(daily_prices)

assert clean_issues == []
print("[PASS] Clean dataset has no basic quality issues")

# Test each invalid price independently
for case_name, bad_price in invalid_cases.items():

    bad_data = daily_prices.copy()

    mask = (
        (bad_data["security_id"] == 1001)
        & (bad_data["date"] == as_of)
    )

    bad_data.loc[mask, "close"] = bad_price

    issues = validate_daily_prices(bad_data)

    print(f"{case_name}: {issues}")

    assert len(issues) > 0

print("[PASS] All invalid-price anomalies detected")


[PASS] Clean dataset has no basic quality issues
Missing price: ['Missing closing prices']
Zero price: ['Non-positive closing prices']
Negative price: ['Non-positive closing prices']
[PASS] All invalid-price anomalies detected


In [17]:
# Day 6 — Step 2.3.1
# Stale Price Experiment

as_of = pd.Timestamp("2021-04-15")

# Construct a dataset where ALPHA has no price observations
# after March 1, 2021.
stale_prices = daily_prices.loc[
    ~(
        (daily_prices["security_id"] == 1001)
        & (daily_prices["date"] > pd.Timestamp("2021-03-01"))
    )
].copy()

# Apply the existing price eligibility function
stale_result = get_pit_price_universe(
    security_master,
    stale_prices,
    as_of_date=as_of,
    min_price=5.0
)

alpha = stale_result.loc[
    stale_result["ticker"] == "ALPHA"
].iloc[0]

print("Selection date:", as_of.date())
print("ALPHA latest price date:", alpha["date"].date())
print("ALPHA latest close:", alpha["close"])
print("Price eligible:", alpha["price_eligible"])

# Demonstrate that the current function accepts stale prices
assert bool(alpha["price_eligible"])

print("[CONFIRMED] Existing function accepts stale prices")

Selection date: 2021-04-15
ALPHA latest price date: 2021-03-01
ALPHA latest close: 100.0
Price eligible: True
[CONFIRMED] Existing function accepts stale prices


In [18]:
# Day 6 — Step 2.3.2
# Price Freshness Check

def add_price_freshness_check(
    universe,
    as_of_date
):
    result = universe.copy()
    as_of_date = pd.Timestamp(as_of_date)

    # The latest price must be from the selection date
    result["price_fresh"] = (
        result["date"].notna()
        & (result["date"] == as_of_date)
    )

    # Add freshness to the existing price-eligibility rule
    result["price_eligible"] = (
        result["price_eligible"]
        & result["price_fresh"]
    )

    return result


checked_result = add_price_freshness_check(
    stale_result,
    as_of_date=as_of
)

display(
    checked_result[
        [
            "ticker",
            "date",
            "close",
            "price_fresh",
            "price_eligible"
        ]
    ]
)

# Verify that stale ALPHA is excluded
alpha_checked = checked_result.loc[
    checked_result["ticker"] == "ALPHA"
].iloc[0]

assert not bool(alpha_checked["price_fresh"])
assert not bool(alpha_checked["price_eligible"])

print("[PASS] Stale ALPHA price correctly rejected")


,ticker,date,close,price_fresh,price_eligible
0,ALPHA,2021-03-01,100.0,False,False
1,GAMMA,2021-04-15,4.0,True,False


[PASS] Stale ALPHA price correctly rejected


In [19]:
# Week 2 Day 7 — Step 1.2
# Corrected Baseline Stale Price Audit

import pandas as pd

as_of = pd.Timestamp("2021-04-15")
last_alpha_date = pd.Timestamp("2021-04-09")

# 1. Clean baseline
baseline = get_pit_tradable_universe(
    security_master,
    daily_prices,
    as_of_date=as_of
)

print("BASELINE — CLEAN DATA")
display(baseline[
    [
        "ticker", "date", "close",
        "observation_count",
        "price_eligible",
        "liquidity_eligible", "eligible"
    ]
])

# 2. Remove only recent ALPHA records
stale_data = daily_prices.loc[
    ~(
        (daily_prices["security_id"] == 1001)
        & (daily_prices["date"] > last_alpha_date)
    )
].copy()

# 3. Check the existing combined function
stale_baseline = get_pit_tradable_universe(
    security_master,
    stale_data,
    as_of_date=as_of
)

print("\nBASELINE — STALE ALPHA DATA")
display(stale_baseline[
    [
        "ticker", "date", "close",
        "observation_count",
        "price_eligible",
        "liquidity_eligible", "eligible"
    ]
])

alpha_stale = stale_baseline.loc[
    stale_baseline["ticker"] == "ALPHA"
].iloc[0]

print("\nALPHA latest price date:", alpha_stale["date"].date())
print("ALPHA observation count:", alpha_stale["observation_count"])
print("ALPHA currently eligible:", bool(alpha_stale["eligible"]))

# 4. Verify the test isolates stale-price risk
assert alpha_stale["date"] == last_alpha_date
assert alpha_stale["observation_count"] == 60
assert bool(alpha_stale["liquidity_eligible"])
assert bool(alpha_stale["eligible"])

print("[CONFIRMED] Stale price accepted despite sufficient liquidity history")

BASELINE — CLEAN DATA


,ticker,date,close,observation_count,price_eligible,liquidity_eligible,eligible
0,ALPHA,2021-04-15,100.0,60,True,True,True
1,GAMMA,2021-04-15,4.0,60,False,True,False



BASELINE — STALE ALPHA DATA


,ticker,date,close,observation_count,price_eligible,liquidity_eligible,eligible
0,ALPHA,2021-04-09,100.0,60,True,True,True
1,GAMMA,2021-04-15,4.0,60,False,True,False



ALPHA latest price date: 2021-04-09
ALPHA observation count: 60
ALPHA currently eligible: True
[CONFIRMED] Stale price accepted despite sufficient liquidity history


In [20]:
# Week 2 Day 7 — Step 2.1
# Integrated Price Freshness — Version 2

def get_pit_tradable_universe_v2(
    security_master,
    daily_prices,
    as_of_date,
    min_price=5.0,
    min_median_dollar_volume=10_000_000,
    window=60,
):
    as_of_date = pd.Timestamp(as_of_date)

    # Reuse the existing historical price and liquidity logic
    result = get_pit_tradable_universe(
        security_master=security_master,
        daily_prices=daily_prices,
        as_of_date=as_of_date,
        min_price=min_price,
        min_median_dollar_volume=min_median_dollar_volume,
        window=window,
    ).copy()

    # Simplified freshness rule for synthetic data:
    # the last observation must be on the selection date.
    result["price_fresh"] = (
        result["date"].notna()
        & (result["date"] == as_of_date)
    )

    # Freshness is mandatory, not an optional manual step
    result["price_eligible"] = (
        result["price_eligible"]
        & result["price_fresh"]
    )

    # Recompute final eligibility
    result["eligible"] = (
        result["price_eligible"]
        & result["liquidity_eligible"]
    )

    return result


In [21]:
# Week 2 Day 7 — Step 2.2
# Old vs New Eligibility Comparison

# Case A: Clean synthetic data
clean_v2 = get_pit_tradable_universe_v2(
    security_master,
    daily_prices,
    "2021-04-15"
)

print("CLEAN DATA — V2")
display(clean_v2[
    [
        "ticker",
        "date",
        "close",
        "price_fresh",
        "price_eligible",
        "liquidity_eligible",
        "eligible",
    ]
])

# Case B: Stale ALPHA data from Step 1
stale_v2 = get_pit_tradable_universe_v2(
    security_master,
    stale_data,
    "2021-04-15"
)

print("\nSTALE DATA — V2")
display(stale_v2[
    [
        "ticker",
        "date",
        "close",
        "price_fresh",
        "price_eligible",
        "liquidity_eligible",
        "eligible",
    ]
])

# Check expected outcomes
clean_eligible = set(
    clean_v2.loc[clean_v2["eligible"], "ticker"]
)
stale_eligible = set(
    stale_v2.loc[stale_v2["eligible"], "ticker"]
)

assert clean_eligible == {"ALPHA"}
assert stale_eligible == set()

alpha_stale_v2 = stale_v2.set_index("ticker").loc["ALPHA"]

assert bool(alpha_stale_v2["liquidity_eligible"])
assert not bool(alpha_stale_v2["price_fresh"])
assert not bool(alpha_stale_v2["eligible"])

print("\n[PASS] Clean data retains ALPHA")
print("[PASS] Stale ALPHA excluded despite sufficient liquidity")


CLEAN DATA — V2


,ticker,date,close,price_fresh,price_eligible,liquidity_eligible,eligible
0,ALPHA,2021-04-15,100.0,True,True,True,True
1,GAMMA,2021-04-15,4.0,True,False,True,False



STALE DATA — V2


,ticker,date,close,price_fresh,price_eligible,liquidity_eligible,eligible
0,ALPHA,2021-04-09,100.0,False,False,True,False
1,GAMMA,2021-04-15,4.0,True,False,True,False



[PASS] Clean data retains ALPHA
[PASS] Stale ALPHA excluded despite sufficient liquidity


In [22]:
# Week 2 Day 7 — Step 3.1
# Extended synthetic dataset for 252-session testing

import pandas as pd

extended_dates = pd.bdate_range(
    start="2020-01-02",
    periods=300
)

extended_records = []

for date in extended_dates:
    for security_id, spec in synthetic_specs.items():

        # GAMMA starts later in this synthetic dataset
        # and therefore has fewer historical observations.
        if security_id == 1003 and date < extended_dates[200]:
            continue

        extended_records.append({
            "date": date,
            "security_id": security_id,
            "close": spec["close"],
            "volume": spec["volume"],
        })

extended_prices = pd.DataFrame(extended_records)

extended_prices = extended_prices.sort_values(
    ["security_id", "date"]
).reset_index(drop=True)

print("Extended dataset shape:", extended_prices.shape)

print("\nObservation counts:")
print(
    extended_prices.groupby("security_id")["date"].count()
)

print("\nFinal synthetic date:", extended_dates[-1].date())


Extended dataset shape: (1300, 4)

Observation counts:
security_id
1001    300
1002    300
1003    100
1004    300
1005    300
Name: date, dtype: int64

Final synthetic date: 2021-02-24


In [23]:
# Week 2 Day 7 — Step 3.2
# Point-in-Time Minimum History Check

def get_pit_history(
    daily_prices,
    as_of_date,
    min_history=252
):
    as_of_date = pd.Timestamp(as_of_date)

    history = daily_prices.loc[
        daily_prices["date"] <= as_of_date
    ].copy()

    # Duplicate records should not inflate history counts
    if history.duplicated(
        ["security_id", "date"]
    ).any():
        raise ValueError(
            "Duplicate security-date records"
        )

    # Only count valid closing prices
    history["valid_close"] = (
        history["close"].notna()
        & (history["close"] > 0)
    )

    summary = (
        history.groupby("security_id")
        .agg(
            history_count=("valid_close", "sum")
        )
        .reset_index()
    )

    summary["history_eligible"] = (
        summary["history_count"] >= min_history
    )

    return summary


history_summary = get_pit_history(
    extended_prices,
    as_of_date=extended_dates[-1],
    min_history=252
)

display(history_summary)


,security_id,history_count,history_eligible
0,1001,300,True
1,1002,300,True
2,1003,100,False
3,1004,300,True
4,1005,300,True


In [24]:
# Week 2 Day 7 — Step 3.3
# Minimum History Boundary Tests

alpha_prices = extended_prices.loc[
    extended_prices["security_id"] == 1001
].sort_values("date")

for n, expected in [
    (251, False),
    (252, True),
    (300, True),
]:
    sample = alpha_prices.head(n).copy()

    result = get_pit_history(
        sample,
        as_of_date=sample["date"].max(),
        min_history=252
    )

    actual = bool(
        result.iloc[0]["history_eligible"]
    )

    print(
        f"{n} observations: "
        f"eligible={actual}"
    )

    assert actual == expected

print("[PASS] All minimum-history boundary tests")


251 observations: eligible=False
252 observations: eligible=True
300 observations: eligible=True
[PASS] All minimum-history boundary tests


In [25]:
# Week 2 Day 7 — Step 3.4.1
# Combined PIT Eligibility Version 3

def get_pit_tradable_universe_v3(
    security_master,
    daily_prices,
    as_of_date,
    min_price=5.0,
    min_median_dollar_volume=10_000_000,
    liquidity_window=60,
    min_history=252,
):
    # Step 1: Historical security, price,
    # liquidity and freshness checks
    result = get_pit_tradable_universe_v2(
        security_master=security_master,
        daily_prices=daily_prices,
        as_of_date=as_of_date,
        min_price=min_price,
        min_median_dollar_volume=min_median_dollar_volume,
        window=liquidity_window,
    ).copy()

    # Step 2: Historical valid-price observation count
    history = get_pit_history(
        daily_prices=daily_prices,
        as_of_date=as_of_date,
        min_history=min_history,
    )

    # Step 3: Merge by stable security identifier
    result = result.merge(
        history,
        on="security_id",
        how="left",
        validate="one_to_one",
    )

    # Missing history must not count as eligible
    result["history_count"] = (
        result["history_count"].fillna(0).astype(int)
    )

    result["history_eligible"] = (
        result["history_eligible"].fillna(False)
    ).astype(bool)

    # Step 4: Recompute final eligibility
    result["eligible"] = (
        result["price_eligible"]
        & result["liquidity_eligible"]
        & result["history_eligible"]
    )

    return result


In [26]:
# Week 2 Day 7 — Step 3.4.2
# Combined V3 Test

final_date = extended_dates[-1]

result_v3 = get_pit_tradable_universe_v3(
    security_master,
    extended_prices,
    as_of_date=final_date,
)

columns_to_show = [
    "ticker",
    "close",
    "observation_count",
    "history_count",
    "price_fresh",
    "price_eligible",
    "liquidity_eligible",
    "history_eligible",
    "eligible",
]

display(result_v3[columns_to_show])

# Only ALPHA should pass
eligible_tickers = set(
    result_v3.loc[result_v3["eligible"], "ticker"]
)

assert eligible_tickers == {"ALPHA"}

alpha = result_v3.set_index("ticker").loc["ALPHA"]
gamma = result_v3.set_index("ticker").loc["GAMMA"]

assert alpha["history_count"] == 300
assert bool(alpha["history_eligible"])
assert bool(alpha["price_fresh"])

assert gamma["history_count"] == 100
assert not bool(gamma["history_eligible"])
assert not bool(gamma["price_eligible"])

print("[PASS] V3 combined eligibility")
print("Final eligible universe:", sorted(eligible_tickers))

,ticker,close,observation_count,history_count,price_fresh,price_eligible,liquidity_eligible,history_eligible,eligible
0,ALPHA,100.0,60,300,True,True,True,True,True
1,GAMMA,4.0,60,100,True,False,True,False,False


[PASS] V3 combined eligibility
Final eligible universe: ['ALPHA']


In [27]:
# Week 2 Day 7 — Step 3.4.3
# Isolated Minimum-History Test

history_test_prices = extended_prices.copy()

# Give GAMMA a valid price.
# Its historical observation count stays at 100.
gamma_mask = (
    history_test_prices["security_id"] == 1003
)

history_test_prices.loc[gamma_mask, "close"] = 50.0

isolated_result = get_pit_tradable_universe_v3(
    security_master,
    history_test_prices,
    as_of_date=final_date,
)

gamma = isolated_result.set_index("ticker").loc["GAMMA"]

print("GAMMA price:", gamma["close"])
print("GAMMA history count:", gamma["history_count"])
print("Price eligible:", gamma["price_eligible"])
print("Liquidity eligible:", gamma["liquidity_eligible"])
print("History eligible:", gamma["history_eligible"])
print("Final eligible:", gamma["eligible"])

assert bool(gamma["price_eligible"])
assert bool(gamma["liquidity_eligible"])
assert not bool(gamma["history_eligible"])
assert not bool(gamma["eligible"])

print("[PASS] Minimum-history rule independently verified")

GAMMA price: 50.0
GAMMA history count: 100
Price eligible: True
Liquidity eligible: True
History eligible: False
Final eligible: False
[PASS] Minimum-history rule independently verified


In [28]:
# Future-Data Invariance Regression Test

# Choose an earlier date so future observations exist
test_as_of = extended_dates[269]

baseline_v3 = get_pit_tradable_universe_v3(
    security_master,
    extended_prices,
    as_of_date=test_as_of,
)

# Deliberately modify all future prices and volumes
future_modified = extended_prices.copy()

future_mask = (
    future_modified["date"] > test_as_of
)

assert future_mask.any()

future_modified.loc[future_mask, "close"] = 9999.0
future_modified.loc[future_mask, "volume"] = 1

modified_v3 = get_pit_tradable_universe_v3(
    security_master,
    future_modified,
    as_of_date=test_as_of,
)

# Compare all resulting fields, not just eligible
pd.testing.assert_frame_equal(
    baseline_v3.reset_index(drop=True),
    modified_v3.reset_index(drop=True),
)

print("[PASS] Future price and volume changes do not alter historical eligibility")

print(
    "Eligible universe:",
    sorted(
        baseline_v3.loc[
            baseline_v3["eligible"], "ticker"
        ]
    )
)

[PASS] Future price and volume changes do not alter historical eligibility
Eligible universe: ['ALPHA']


In [29]:
# Week 2 Day 7 — Step 4.2
# Stale Price Regression Test for V3

test_as_of = extended_dates[-1]

# Remove the final three ALPHA observations
stale_extended = extended_prices.loc[
    ~(
        (extended_prices["security_id"] == 1001)
        & (
            extended_prices["date"]
            > extended_dates[-4]
        )
    )
].copy()

stale_v3 = get_pit_tradable_universe_v3(
    security_master,
    stale_extended,
    as_of_date=test_as_of,
)

alpha = stale_v3.set_index("ticker").loc["ALPHA"]

print("ALPHA history count:", alpha["history_count"])
print("ALPHA liquidity eligible:", alpha["liquidity_eligible"])
print("ALPHA price fresh:", alpha["price_fresh"])
print("ALPHA final eligible:", alpha["eligible"])

# History and liquidity remain sufficient
assert alpha["history_count"] == 297
assert bool(alpha["history_eligible"])
assert bool(alpha["liquidity_eligible"])

# Freshness must independently exclude ALPHA
assert not bool(alpha["price_fresh"])
assert not bool(alpha["eligible"])

print("[PASS] V3 excludes stale prices independently")


ALPHA history count: 297
ALPHA liquidity eligible: True
ALPHA price fresh: False
ALPHA final eligible: False
[PASS] V3 excludes stale prices independently


In [30]:
# Week 2 Day 7 — Step 4.3
# Duplicate Data Regression Test

duplicate_prices = extended_prices.copy()

duplicate_prices = pd.concat(
    [
        duplicate_prices,
        extended_prices.iloc[[0]].copy()
    ],
    ignore_index=True
)

try:
    get_pit_tradable_universe_v3(
        security_master,
        duplicate_prices,
        as_of_date=extended_dates[-1],
    )
except ValueError as exc:
    assert "Duplicate security-date records" in str(exc)
    print("[PASS] V3 rejects duplicate security-date records")
else:
    raise AssertionError(
        "V3 failed to reject duplicate records"
    )


[PASS] V3 rejects duplicate security-date records
